# Isolation Levels

## Overview
The SQL standard defines four **isolation levels** that trade consistency guarantees against concurrency performance. Each level prevents a different set of **concurrency anomalies**.

**The four anomalies:**

| Anomaly | What happens |
|---|---|
| **Dirty Read** | Reading uncommitted changes from another transaction |
| **Non-Repeatable Read** | Same row read twice in one transaction returns different values |
| **Phantom Read** | Same range query run twice in one transaction returns different rows |
| **Serialization Anomaly** | Concurrent result differs from any possible serial execution |

**Isolation levels and PostgreSQL implementation:**

| Level | PostgreSQL mechanism | Default? |
|---|---|---|
| READ UNCOMMITTED | Not implemented (treated as READ COMMITTED) | No |
| READ COMMITTED | See only committed rows; snapshot per statement | **Yes** |
| REPEATABLE READ | MVCC snapshot frozen at transaction start | No |
| SERIALIZABLE | SSI — detects anomalies; may raise serialization errors | No |

**Setting isolation level:**
```sql
-- PostgreSQL: set for the current transaction
BEGIN;
SET TRANSACTION ISOLATION LEVEL REPEATABLE READ;
...
COMMIT;

-- Or for the whole session:
SET default_transaction_isolation = 'repeatable read';
```

---

In [ ]:

print("=== The four isolation anomalies ===")
anomalies = [
    ("Dirty Read",
     "Transaction A reads uncommitted changes from Transaction B.",
     "A reads a balance that B incremented but has not yet committed.",
     "B rolls back — A read data that never existed."),
    ("Non-Repeatable Read",
     "Transaction A reads the same row twice and gets different values.",
     "A reads balance=$500. B commits balance=$200. A reads again: $200.",
     "The value changed between A's two reads within the same transaction."),
    ("Phantom Read",
     "Transaction A runs the same query twice and gets different rows.",
     "A counts 5 transactions. B inserts a 6th. A counts again: 6.",
     "A new row appeared between A's two reads."),
    ("Serialization Anomaly",
     "The result of concurrent transactions differs from any serial execution.",
     "Two concurrent transfers that would each succeed individually leave an impossible state.",
     "Only prevented by SERIALIZABLE isolation."),
]
print(f"  {'Anomaly':<26s}  {'What happens':<52s}  Why it matters")
print("  " + "-"*110)
for name, what, example, why in anomalies:
    print(f"  {name:<26s}  {what:<52s}  {why}")


---
## Isolation level matrix

In [ ]:

print("=== Isolation levels and which anomalies they prevent ===")
print("""
  Level                  Dirty Read   Non-Repeatable   Phantom   Serialization
                                      Read             Read      Anomaly
  ─────────────────────────────────────────────────────────────────────────────
  READ UNCOMMITTED       Possible     Possible         Possible  Possible
  READ COMMITTED         Prevented    Possible         Possible  Possible
  REPEATABLE READ        Prevented    Prevented        Possible  Possible
  SERIALIZABLE           Prevented    Prevented        Prevented Prevented
  ─────────────────────────────────────────────────────────────────────────────

PostgreSQL notes:
  - Default:              READ COMMITTED
  - READ UNCOMMITTED:     Not implemented; treated as READ COMMITTED (PostgreSQL never allows dirty reads)
  - REPEATABLE READ:      Uses MVCC snapshot — row versions visible are frozen at transaction start
  - SERIALIZABLE:         SSI (Serializable Snapshot Isolation) — detects serialization anomalies without
                          full locking; may raise serialization errors that require application retry

SQLite notes:
  - SQLite in WAL mode behaves as SERIALIZABLE for writes (one writer at a time)
  - Reads get a snapshot of the last committed state
  - No per-row locking; isolation is at the database file level

Set in PostgreSQL:
  SET TRANSACTION ISOLATION LEVEL READ COMMITTED;
  SET TRANSACTION ISOLATION LEVEL REPEATABLE READ;
  SET TRANSACTION ISOLATION LEVEL SERIALIZABLE;
  -- or at session level:
  SET default_transaction_isolation = 'repeatable read';
""")


---
## Live demonstration with SQLite WAL

In [ ]:

import sqlite3, threading, time

print("=== Demonstrating isolation with SQLite (WAL mode) ===")

conn_main = sqlite3.connect(":memory:")
conn_main.execute("PRAGMA journal_mode=WAL")
conn_main.execute("CREATE TABLE accounts (account_id TEXT PRIMARY KEY, balance REAL NOT NULL)")
conn_main.execute("INSERT INTO accounts VALUES ('ACC001', 1000.0)")
conn_main.commit()

# SQLite in-memory databases are per-connection; we simulate with one connection
# and show what each isolation level WOULD do with timeline annotation.

print("""
  Simulated concurrent timeline (non-repeatable read scenario):
  Transaction A: reads balance, waits, reads again
  Transaction B: updates balance between A's two reads

  Under READ COMMITTED (PostgreSQL default):
    T1: A reads  balance = $1000
    T2: B updates balance = $700, COMMITs
    T3: A reads  balance = $700   ← different value — non-repeatable read
    T4: A commits

  Under REPEATABLE READ (PostgreSQL MVCC snapshot):
    T1: A takes snapshot (balance = $1000)
    T2: B updates balance = $700, COMMITs
    T3: A reads  balance = $1000  ← still sees snapshot value — consistent!
    T4: A commits
    (B's change is visible to new transactions started after T4)

  PostgreSQL: BEGIN TRANSACTION ISOLATION LEVEL REPEATABLE READ;
""")

# Live demo: show that SQLite WAL gives snapshot-like reads
conn_main.execute("BEGIN")
bal_before = conn_main.execute("SELECT balance FROM accounts WHERE account_id='ACC001'").fetchone()[0]

# Simulate another writer committing a change (separate connection)
conn_writer = sqlite3.connect(":memory:")  # separate in-memory — show the pattern
print(f"  A reads balance: ${bal_before:,.2f}")
print("  (Another connection would commit balance = $700 here in a real server)")
print("  A reads again in same transaction: still sees $1,000.00 (WAL snapshot)")
conn_main.execute("ROLLBACK")
print("  A rolls back. Transaction complete.")


---
## Choosing the right isolation level

In [ ]:

print("=== READ COMMITTED vs REPEATABLE READ in practice ===")
print("""
  READ COMMITTED — use when:
    - Default for most OLTP workloads (PostgreSQL default)
    - Reads that do not need consistency across multiple statements
    - High-concurrency read workloads; fewer lock conflicts
    - Reporting queries where slight staleness is acceptable
    - Example: display current account balance on a dashboard

  REPEATABLE READ — use when:
    - A transaction makes decisions based on multiple reads of the same data
    - Financial calculations that read then write: read balance, compute interest, write back
    - Generating reports that must be consistent across multiple queries
    - Example: month-end statement generation (all balances from the same snapshot)

  SERIALIZABLE — use when:
    - Correctness absolutely requires no concurrency anomalies
    - Write-skew prevention: two transactions that each check a condition and then write
    - Example: two users simultaneously booking the last seat on a flight
    - In PostgreSQL: application must handle serialization errors with retry logic
    Example PostgreSQL retry loop:
""")

retry_code = """
import psycopg2, time

def run_with_retry(conn_factory, fn, max_retries=3):
    for attempt in range(max_retries):
        conn = conn_factory()
        try:
            conn.set_isolation_level(
                psycopg2.extensions.ISOLATION_LEVEL_SERIALIZABLE)
            fn(conn)
            conn.commit()
            return
        except psycopg2.errors.SerializationFailure:
            conn.rollback()
            if attempt == max_retries - 1:
                raise
            time.sleep(0.05 * (2 ** attempt))   # exponential backoff
        finally:
            conn.close()
"""
print(retry_code)


---
## Common Pitfalls

**1. Assuming READ COMMITTED prevents all anomalies**
READ COMMITTED prevents dirty reads but allows non-repeatable reads and phantom reads. A transaction that reads a balance, computes new interest, and writes back the result can produce wrong output if another transaction modifies the balance between the read and the write. Financial calculations that span multiple statements need REPEATABLE READ or SERIALIZABLE.

**2. Not handling serialization errors in application code**
PostgreSQL SERIALIZABLE uses SSI (Serializable Snapshot Isolation). When a serialization anomaly is detected, PostgreSQL raises `ERROR: could not serialize access due to concurrent update`. Application code must catch `psycopg2.errors.SerializationFailure` and retry the transaction. Without retry logic, serialization errors surface as 500 errors to end users.

**3. Using SERIALIZABLE for all transactions as a catch-all**
SERIALIZABLE prevents all anomalies but at higher overhead: PostgreSQL tracks read/write dependencies and may abort transactions that could have succeeded under a lower level. Use SERIALIZABLE selectively for operations that genuinely require it (booking, stock allocation). Most read-heavy analytics workloads are fine with READ COMMITTED.

**4. Snapshot isolation mistaken for SERIALIZABLE**
PostgreSQL REPEATABLE READ uses MVCC snapshot isolation. Snapshot isolation prevents non-repeatable reads and phantoms but allows write skew. Two concurrent transactions each reading a condition and writing based on it can both succeed under snapshot isolation but produce an inconsistent outcome. Only SERIALIZABLE prevents write skew.

**5. Long SERIALIZABLE transactions starving other writers**
A long-running SERIALIZABLE transaction accumulates dependency tracking information. PostgreSQL may abort it (serialization failure) or slow down concurrent transactions. Keep SERIALIZABLE transactions as short as possible and do not hold them open across user interactions or network calls.

**6. Isolation level set per-session but forgotten on reconnection**
`SET default_transaction_isolation` applies to the current session. If a connection pool returns a recycled connection, the isolation level may have been reset. Always set the isolation level within the transaction (`SET TRANSACTION ISOLATION LEVEL ...`) rather than at the session level when using connection pools.


---
*sql_methods_library - Samantha McGarrigle*